Attention mechanism
- solves RNN bottleneck, allowing model to process sequences without reliance on recurrence
- allows each element to consider and weight every other elements

In [8]:
import torch 
import torch.nn as nn

Motivation
- tokenizing is important for machines to understand language
- turning words into numerical vectors
- but one word can have diff meanings and context
- hence we want word embeddings with context understanding

Attention head
- a component in neural network that takes in the initial embeding vectors, return a seq of recalculated embedding with respect to all other words
- multi head attn -> each head captures a set of relationship (abstract in reality)
- combining the outputs of all head returns same size embedding vector
- dmodel -> size of embedding vectore of input, same as the size of output after attention

In [9]:
batch_size, dmodel = 1,12
seq_en_len, seq_de_len = 4,5

# how to calculte alignment -> how similar are 2 elements in seq
# using scaled dot product '
input_en = torch.rand(batch_size, seq_en_len, dmodel)
input_de = torch.rand(batch_size, seq_de_len, dmodel)

'''
Transformers do not use initial embedding vectors to calculate alignment as same words
can have diff meaning but same embeding
They use 3 embeding space, Q K V
Attn uses 3 weights Wk Wq Wv to convert initial embeddings to respective
Q K V vectors -> the q k v are of lower dim compared to d model
Wq = dmodel x dq
Weights matrices are learned, weights get updated

muliply Q Kt to get alignment scores, softmax per row so they sum to 1
-> normalize the attn scores
multiply attn scores with V and sum up to get final embedding
divide qkt by root dk to stablise ur values during training as large 
dot products can produce large exp in sfotmax, -> small gradients -> harder traininbg

'''

'\nTransformers do not use initial embedding vectors to calculate alignment as same words\ncan have diff meaning but same embeding\nThey use 3 embeding space, Q K V\nAttn uses 3 weights Wk Wq Wv to convert initial embeddings to respective\nQ K V vectors -> the q k v are of lower dim compared to d model\nWq = dmodel x dq\nWeights matrices are learned, weights get updated\n\nmuliply Q Kt to get alignment scores, softmax per row so they sum to 1\n-> normalize the attn scores\nmultiply attn scores with V and sum up to get final embedding\ndivide qkt by root dk to stablise ur values during training as large \ndot products can produce large exp in sfotmax, -> small gradients -> harder traininbg\n\n'

In [10]:
class Attention(nn.Module):
    
    def __init__(self):
        super().__init__
    
    def forward(self, Q,K,V):
        out = torch.matmul(Q,K.transpose(1,2))
        out = out / (K.shape[-1]**0.5)
        
        out = nn.Softmax(out,dim=-1)
        
        return torch.matmul(out,V)

attention in decoder
- same as attn in encoder, but with cross attn to encoder outputs as well
- self attn first then decoder
- the q in cross attn is from self attn outputs after recalculation

Attn head
- network component that does the above, transform embed to q k v and use weight to get scaled dot product atn

In [11]:
class AttentionHead(nn.Module):
    
    def __init__(self, model_size, qkv_size):
        super().__init__
        self.Wq = nn.Linear(model_size,qkv_size)
        self.Wk = nn.Linear(model_size,qkv_size)
        self.Wv = nn.Linear(model_size,qkv_size)
        self.attention = Attention()
    
    def forward(self,query,key,value):
        self.attention(self.Wq(query),self.Wk(key),self.Wv(value))


In [13]:
qkv_size = 6

encoder_self_attn = AttentionHead(dmodel, qkv_size)
decoder_self_attn = AttentionHead(dmodel, qkv_size)

encoder_self_attn_out = encoder_self_attn(input_en,input_en,input_en)
decoder_self_attn_out = decoder_self_attn(input_de,input_de,input_de)

decoder_cross_attn = AttentionHead(dmodel, qkv_size)
decoder_cross_attention_out = decoder_cross_attn(input_de, input_en, input_en)



AttributeError: cannot assign module before Module.__init__() call

Multihead attn
- attend to diff properties of input
- compute in parallel
- sizes of each head smaller than original head dk=dmodel/h

the output of all heads concat together
- then multiplied by Wo (trainable param) to get final output

In [ ]:
class MultiheadAttention(nn.Module):
    
    def __init__(self, dmodel, num_heads):
        super().__init__
        self.qkv_size = dmodel/num_heads
        
        self.heads = nn.ModuleList([
            AttentionHead(dmodel,self.qkv_size) for _ in range(num_heads)
        ])
        
        self.Wo = nn.Linear(dmodel,dmodel)
    
    def forward(self,query,key,value):
        out_heads = tuple([attention_head(query,key,value) for attention_head in self.heads])
        
        out = torch.cat(out_heads,dim=-1)
        return self.Wo(out)